In [ ]:
from pathlib import Path
import io
import zipfile
import math
import textwrap
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.patches import Rectangle
from scipy.ndimage import gaussian_filter
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["figure.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9


In [ ]:
BASE = Path("/content") if IN_COLAB else Path("/mnt/data")
WORK = BASE / "figure1_reproducible_work"
WORK.mkdir(parents=True, exist_ok=True)
INPUT_ZIP = BASE / "painting_geometry_phase4_artbench_pilot.zip"
ARTBENCH_ROOT = BASE / "artbench_data"
FIG1_IMAGE_DIR = BASE / "figure1_images"
FIG1_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUTDIR = WORK / "output"
OUTDIR.mkdir(parents=True, exist_ok=True)
DESCRIPTOR_COL = "geom__curv__kappa_ref_s2p0_grad_weighted_abs"
FIG1_SELECTION = [
    {"role": "Low geometry", "style": "post_impressionism", "artist_slug": "anita-malfatti", "artist_label": "Anita Malfatti", "title": "Fernanda de Castro", "year": "1922", "filename": "anita-malfatti_fernanda-de-castro-1922.jpg", "expected_value": 0.252172},
    {"role": "Intermediate geometry", "style": "post_impressionism", "artist_slug": "amrita-sher-gil", "artist_label": "Amrita Sher-Gil", "title": "Tribal Women", "year": "1938", "filename": "amrita-sher-gil_tribal-women-1938.jpg", "expected_value": 0.324556},
    {"role": "High geometry", "style": "post_impressionism", "artist_slug": "abraham-manievich", "artist_label": "Abraham Manievich", "title": "The Yellow House", "year": "", "filename": "abraham-manievich_the-yellow-house.jpg", "expected_value": 0.403162},
]
EXPORT_BASENAME = "Figure1_multiscale_luminance_geometry_no_panel_d"
PNG_DPI = 600


In [ ]:
def resize_long_side(pil_img, long_side=256):
    w, h = pil_img.size
    scale = float(long_side) / float(max(w, h))
    new_size = (max(1, round(w * scale)), max(1, round(h * scale)))
    return pil_img.resize(new_size, Image.Resampling.LANCZOS)
def luminance_bt601(arr_rgb):
    arr = np.asarray(arr_rgb, dtype=float)
    y = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]
    y -= y.min()
    denom = y.max() - y.min()
    return np.zeros_like(y) if denom <= 0 else y / denom
def curvature_dog_scale_normalized(I, sigma_ref, long_side, reference_long_side=512, eps=1e-12, grad_quantile=0.20):
    sigma_px = float(sigma_ref) * float(long_side) / float(reference_long_side)
    kw = dict(sigma=sigma_px, mode="reflect", truncate=3.0)
    Ix  = gaussian_filter(I, order=(0, 1), **kw)
    Iy  = gaussian_filter(I, order=(1, 0), **kw)
    Ixx = gaussian_filter(I, order=(0, 2), **kw)
    Iyy = gaussian_filter(I, order=(2, 0), **kw)
    Ixy = gaussian_filter(I, order=(1, 1), **kw)
    grad2 = Ix * Ix + Iy * Iy
    grad = np.sqrt(grad2)
    kappa = (Ixx * Iy * Iy - 2.0 * Ix * Iy * Ixy + Iyy * Ix * Ix) / np.power(grad2 + eps * eps, 1.5)
    finite = np.isfinite(kappa) & np.isfinite(grad)
    pos = grad[finite & (grad > 0)]
    threshold = np.quantile(pos, grad_quantile) if pos.size else 0.0
    valid = finite & (grad >= threshold)
    return sigma_px * kappa, grad, valid
def panel_letter(ax, s):
    ax.text(-0.06, 1.04, s, transform=ax.transAxes, ha="left", va="bottom", fontsize=20, fontweight="bold")
def read_csv_from_phase4_zip(zip_path, inner_name="artbench_pilot_features.csv"):
    with zipfile.ZipFile(zip_path, "r") as zf:
        matches = [n for n in zf.namelist() if n.endswith(inner_name)]
        if not matches:
            raise FileNotFoundError(f"{inner_name} not found inside {zip_path}")
        with zf.open(matches[0]) as f:
            return pd.read_csv(f)
def resolve_image_path(filename):
    for p in [FIG1_IMAGE_DIR / filename, BASE / filename]:
        if p.exists():
            return p
    if ARTBENCH_ROOT.exists():
        hits = list(ARTBENCH_ROOT.rglob(filename))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"Could not find {filename}. Upload it to {FIG1_IMAGE_DIR} or place it under {ARTBENCH_ROOT}.")
def clean_axes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)


In [ ]:
assert INPUT_ZIP.exists(), f"Missing input ZIP: {INPUT_ZIP}"
phase4_features = read_csv_from_phase4_zip(INPUT_ZIP)
selected_rows = []
for spec in FIG1_SELECTION:
    hit = phase4_features[(phase4_features["style"].astype(str) == spec["style"]) & (phase4_features["artist"].astype(str) == spec["artist_slug"]) & (phase4_features["filename"].astype(str) == spec["filename"])]
    if len(hit) != 1:
        raise RuntimeError(f"Expected exactly one row for {spec['filename']} but found {len(hit)}")
    row = hit.iloc[0].to_dict()
    value = float(row[DESCRIPTOR_COL])
    row.update({"expected_value": spec["expected_value"], "abs_diff": abs(value-spec["expected_value"]), "role": spec["role"], "artist_label": spec["artist_label"], "title": spec["title"], "year": spec["year"]})
    selected_rows.append(row)
selected_df = pd.DataFrame(selected_rows)
print(selected_df[["role","artist_label","title","filename",DESCRIPTOR_COL,"expected_value","abs_diff"]])


In [ ]:
missing = []
for spec in FIG1_SELECTION:
    try:
        resolve_image_path(spec["filename"])
    except FileNotFoundError:
        missing.append(spec["filename"])
if missing and IN_COLAB:
    print("Upload the missing Figure-1 images:")
    print("\n".join(missing))
    uploaded = files.upload()
    for name, payload in uploaded.items():
        target = FIG1_IMAGE_DIR / name
        target.write_bytes(payload)
records = []
for spec in FIG1_SELECTION:
    img_path = resolve_image_path(spec["filename"])
    pil = resize_long_side(Image.open(img_path).convert("RGB"), long_side=256)
    rgb = np.asarray(pil)
    lum = luminance_bt601(rgb)
    records.append({**spec, "path": str(img_path), "pil": pil, "rgb": rgb, "lum": lum})
for r in records: print(r["role"], "->", r["path"], "| shape:", r["rgb"].shape)


In [ ]:
mid = [r for r in records if r["role"].startswith("Intermediate")][0]
mid_lum = mid["lum"]
long_side = max(mid_lum.shape)
sigma_refs = [1,2,4,8]
curv_maps, valid_masks = {}, {}
for s in sigma_refs:
    kappa, grad, valid = curvature_dog_scale_normalized(mid_lum, sigma_ref=s, long_side=long_side, reference_long_side=512, grad_quantile=0.20)
    curv_maps[s], valid_masks[s] = kappa, valid


In [ ]:
fig = plt.figure(figsize=(14.4,10.2), constrained_layout=False)
gs = gridspec.GridSpec(3,12,figure=fig,height_ratios=[1.08,1.00,1.12],hspace=0.34,wspace=0.18)
fig.suptitle("Figure 1. From painting to multiscale luminance geometry",x=0.03,y=0.988,ha="left",va="top",fontsize=19,fontweight="bold")
fig.text(0.03,0.951,"Within one style category, paintings occupy markedly different positions in the level-set geometry measured at an intermediate spatial scale.",ha="left",fontsize=10.8,color="dimgray")
for j,r in enumerate(records):
    ax=fig.add_subplot(gs[0,4*j:4*(j+1)])
    if j==0: panel_letter(ax,"a")
    ax.imshow(r["rgb"]); clean_axes(ax)
    year=f" ({r['year']})" if r["year"] else " (n.d.)"
    validated=selected_df[selected_df["filename"]==r["filename"]][DESCRIPTOR_COL].iloc[0]
    ax.set_title(f"{r['role']}\n{r['artist_label']}, {r['title']}{year}\n"+rf"$G_{{\sigma=2}}={validated:.3f}$",pad=6,fontsize=9.8)
for j,r in enumerate(records):
    ax=fig.add_subplot(gs[1,4*j:4*(j+1)])
    if j==0: panel_letter(ax,"b")
    lum=r["lum"]; ax.imshow(lum,cmap="gray",vmin=0,vmax=1,alpha=0.92)
    ax.contour(lum,levels=[0.15,0.30,0.45,0.60,0.75,0.90],colors="white",linewidths=0.85,alpha=0.95)
    clean_axes(ax); ax.set_title(f"Iso-luminance contours — {r['role'].replace(' geometry','').lower()}",pad=4,fontsize=9.8)
sub_c=gridspec.GridSpecFromSubplotSpec(1,4,subplot_spec=gs[2,0:12],wspace=0.06)
all_valid_abs=[np.abs(curv_maps[s][valid_masks[s]]) for s in sigma_refs if np.any(valid_masks[s])]
shared_clip=np.percentile(np.concatenate(all_valid_abs),99.0)
axs_c=[]; ims=[]
for j,s in enumerate(sigma_refs):
    ax=fig.add_subplot(sub_c[0,j])
    if j==0: panel_letter(ax,"c")
    ghost=np.clip(0.86*mid_lum+0.14,0,1); ax.imshow(ghost,cmap="gray",vmin=0,vmax=1)
    overlay=np.ma.masked_where(~valid_masks[s],curv_maps[s])
    im=ax.imshow(overlay,cmap="coolwarm",vmin=-shared_clip,vmax=shared_clip,alpha=0.78)
    ims.append(im); clean_axes(ax); ax.set_title(rf"$\sigma_{{ref}}={s}$",pad=5,fontsize=10.5); axs_c.append(ax)
left,right=axs_c[0].get_position().x0,axs_c[-1].get_position().x1
bottom=min(a.get_position().y0 for a in axs_c)-0.045
cax=fig.add_axes([left+0.08,bottom,(right-left)-0.16,0.016])
cb=plt.colorbar(ims[0],cax=cax,orientation="horizontal")
cb.set_ticks([-shared_clip,0.0,shared_clip]); cb.set_ticklabels(["negative curvature","0","positive curvature"]); cb.outline.set_linewidth(0.6); cb.ax.tick_params(labelsize=8.5,length=0)
fig.subplots_adjust(left=0.045,right=0.985,top=0.90,bottom=0.10)
png_path=OUTDIR/f"{EXPORT_BASENAME}.png"; pdf_path=OUTDIR/f"{EXPORT_BASENAME}.pdf"; svg_path=OUTDIR/f"{EXPORT_BASENAME}.svg"
fig.savefig(png_path,dpi=PNG_DPI,bbox_inches="tight",facecolor="white"); fig.savefig(pdf_path,bbox_inches="tight",facecolor="white"); fig.savefig(svg_path,bbox_inches="tight",facecolor="white")
plt.show(); print("Saved:",png_path,pdf_path,svg_path,sep="\n - " )
